# 📘 Notebook 05: Files and Formats

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module B: Beyond Tables · Notebook 1 of 2 in this module · (5 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain the difference between storing a table **row by row** and **column by column**
- Show by measurement which layout suits which kind of work
- State the weaknesses of **CSV** as a format for data
- Write and read **Parquet** files, and read only the columns and rows a query needs
- Explain, and implement by hand, the two encodings that make columnar files small: **dictionary** and **run-length** encoding
- Use file **statistics** and **partitioning** to avoid reading data at all

> **Prerequisites:** Notebooks 01 to 04. Python lists and dictionaries.
>
> 🔭 **Why this notebook matters:** a table is an idea. On a disk there are only bytes in a row, and somebody has to decide in what order to write them. That decision turns out to matter enormously. The analytical query of Notebook 04, *total revenue by category*, needs two columns out of a dozen. A file laid out the right way lets us read just those two and skip the rest, and can make the same question ten or a hundred times cheaper. Every modern system for large-scale analysis, including Spark in Module C, stores its data this way.

> 📦 **Libraries used in this notebook.**
>
> - **PyArrow** is the Python interface to Apache Arrow, an open-source project that defines a standard way of holding tables in memory column by column. It also reads and writes the **Parquet** file format, which is what we use it for here. Parquet was created in 2013 by engineers at Twitter and Cloudera and has become the standard file format for analytical data.
> - **Matplotlib**, for one chart.
>
> The cell below installs what is needed. In Google Colab these libraries are already present and the cell finishes in a moment.

In [ ]:
%pip install -q pyarrow matplotlib

In [ ]:
import csv
import os
import random
import datetime
import tempfile
import time

import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

folder = tempfile.mkdtemp()          # a scratch folder for the files of this notebook
print("Files will be written to", folder)

The data is the flattened sales table of Notebook 04: one row per order line, with the descriptions of its date, customer and product written out. A quarter of a million rows and ten columns.

In [ ]:
def make_sales(customers=5000, products=200, orders=100000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    customer_city = [rng.choice(cities) for i in range(customers + 1)]
    product_info = [(f"Book {i}", rng.choice(categories), round(rng.uniform(5, 60), 2)) for i in range(products + 1)]

    rows = []
    first_day = datetime.date(2023, 1, 1)
    for order_id in range(1, orders + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        customer = rng.randint(1, customers)
        for line in range(rng.randint(1, 4)):
            title, category, price = product_info[rng.randint(1, products)]
            quantity = rng.randint(1, 3)
            rows.append((order_id, day.isoformat(), day.year, day.month, customer, customer_city[customer],
                         title, category, quantity, round(quantity * price, 2)))
    rows.sort(key=lambda row: row[1])                      # in order of date, as sales arrive
    return rows

column_names = ["order_id", "date", "year", "month", "customer", "city", "title", "category", "quantity", "revenue"]
rows = make_sales()

print(f"{len(rows):,} rows, {len(column_names)} columns")
print(column_names)
print(rows[0])

## 1. Rows or columns?

### Intuition first
Think of a class register on paper. You can write it **by student**: one line per student, with their name, year, and mark in every subject. Or you can write it **by subject**: one page per subject, listing every student's mark in that subject.

To look up everything about one student, the first is ideal: one line. To compute the class average in mathematics, the second is ideal: one page, with nothing on it but the numbers you need.

A table in a computer faces the same choice.

```
 Row layout (one record after another)          Column layout (one column after another)

 1, 2023-01-01, Athens, Fiction, 2, 25.00       order_id:  1, 2, 3, ...
 2, 2023-01-01, Patras, Science, 1, 41.50       date:      2023-01-01, 2023-01-01, 2023-01-02, ...
 3, 2023-01-02, Athens, Travel,  3, 54.00       city:      Athens, Patras, Athens, ...
 ...                                            revenue:   25.00, 41.50, 54.00, ...
```

Our `rows` list is the row layout. Let us build the column layout from it: a dictionary with one list per column.

In [ ]:
columns = {name: [row[i] for row in rows] for i, name in enumerate(column_names)}

print(type(rows), "of", len(rows), "tuples")
print(type(columns), "of", len(columns), "lists, each of", len(columns["revenue"]), "values")

### An analytical question: add up one column
The total revenue needs one column out of ten.

In [ ]:
def best_of(function, repeats=5):
    timings = []
    for i in range(repeats):
        started = time.perf_counter()
        result = function()
        timings.append(time.perf_counter() - started)
    return result, min(timings) * 1000

revenue_position = column_names.index("revenue")

total_from_rows, time_rows = best_of(lambda: sum(row[revenue_position] for row in rows))
total_from_columns, time_columns = best_of(lambda: sum(columns["revenue"]))

print(f"Row layout:    {time_rows:6.2f} ms")
print(f"Column layout: {time_columns:6.2f} ms   ({time_rows / time_columns:.0f} times faster)")
print("Same total:", round(total_from_rows, 2) == round(total_from_columns, 2))

In the row layout the program has to visit every record and step past nine values it does not want in order to reach the tenth. In the column layout the numbers it wants are sitting next to each other and nothing else is touched.

### A transactional question: fetch whole records
Now the opposite kind of work: show everything about one particular order line, many times over.

In [ ]:
random.seed(0)
positions = [random.randrange(len(rows)) for i in range(50000)]

record_from_rows, fetch_rows = best_of(lambda: [rows[i] for i in positions])
record_from_columns, fetch_columns = best_of(lambda: [tuple(columns[name][i] for name in column_names) for i in positions])

print(f"Row layout:    {fetch_rows:6.2f} ms")
print(f"Column layout: {fetch_columns:6.2f} ms   ({fetch_columns / fetch_rows:.0f} times slower)")
print("Same records:", record_from_rows == record_from_columns)

The result is reversed. In the row layout a record is already in one piece. In the column layout it has to be assembled from ten different places.

| | Read a few columns of every row | Read every column of a few rows |
|---|---|---|
| Typical of | analysis: totals, averages, groupings | transactions: one order, one customer |
| Row layout | must step over the unwanted values | the record is in one piece |
| Column layout | reads only what is needed | must collect the record from many places |
| Better choice | **columns** | **rows** |

Neither layout is better in general. The databases of Module A, built for transactions, store rows. Systems built for analysis store columns. The rest of this notebook is about analysis, and so about columns.

## 2. CSV: the lowest common denominator

The most familiar file format for a table is **CSV**, comma-separated values: plain text, one line per row.

In [ ]:
csv_path = os.path.join(folder, "sales.csv")

with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(column_names)
    writer.writerows(rows)

print(f"Size: {os.path.getsize(csv_path) / 1_000_000:.1f} MB\n")
with open(csv_path, encoding="utf-8") as file:
    for i in range(4):
        print(file.readline().rstrip())

CSV has real virtues. A person can read it. Every program on earth can open it. It needs no special software. For exchanging a small table between two people it is hard to beat.

For holding data that will be analysed, it has three serious weaknesses. Read the file back and look at what arrives.

In [ ]:
with open(csv_path, encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader)
    first = next(reader)

print(first)
print([type(value).__name__ for value in first])

**1. The types are gone.** Every value is a string. The file does not say that `quantity` is a whole number, `revenue` a decimal and `date` a date. Whoever reads it must guess, and different programs guess differently. Is `01/02/2024` the first of February or the second of January? Is `00420` a number, or a postcode whose leading zeros matter?

**2. It is a row layout.** To add up one column, the whole file must be read and every line split at its commas.

**3. It is bulky.** Numbers are written out as text, and the word `Thessaloniki` is spelled in full thousands of times.

In [ ]:
def total_revenue_from_csv():
    total = 0.0
    with open(csv_path, encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        for line in reader:
            total += float(line[revenue_position])
    return total

csv_total, csv_time = best_of(total_revenue_from_csv, repeats=2)
print(f"Total revenue from the CSV file: {csv_total:,.2f}   in {csv_time:.0f} ms")

## 3. Parquet: a columnar file

### Formal definition
**Parquet** is a file format that stores a table **column by column**, together with a **schema** that records the name and type of every column. Inside the file each column is cut into chunks, each chunk is **encoded and compressed** on its own, and a block of **metadata** at the end describes where everything is and what it contains.

Writing one takes two lines. PyArrow builds a table from our dictionary of columns, working out the types as it goes.

In [ ]:
table = pa.table(columns)
print(table.schema, "\n")

parquet_path = os.path.join(folder, "sales.parquet")
pq.write_table(table, parquet_path)

csv_size = os.path.getsize(csv_path) / 1_000_000
parquet_size = os.path.getsize(parquet_path) / 1_000_000
print(f"CSV:     {csv_size:5.1f} MB")
print(f"Parquet: {parquet_size:5.1f} MB   ({csv_size / parquet_size:.0f} times smaller)")

The same data in a fraction of the space, and the types are recorded: `quantity` is an integer and `revenue` a floating-point number, and they will still be when the file is read next year by a different program in a different language.

### Reading only what is needed
Now the analytical question again. We ask for a single column, and time the same request against the CSV file using the same library, so that the comparison is fair.

In [ ]:
whole_csv, time_csv = best_of(lambda: pacsv.read_csv(csv_path), repeats=3)
whole_parquet, time_parquet = best_of(lambda: pq.read_table(parquet_path), repeats=3)
one_column, time_one_column = best_of(lambda: pq.read_table(parquet_path, columns=["revenue"]), repeats=3)

print(f"CSV, whole file (it has no other way): {time_csv:7.1f} ms")
print(f"Parquet, whole file:                   {time_parquet:7.1f} ms")
print(f"Parquet, the revenue column only:      {time_one_column:7.1f} ms")

import pyarrow.compute as pc
print("\nTotal revenue:", f"{pc.sum(one_column['revenue']).as_py():,.2f}")

Reading one column from the Parquet file is many times faster than reading the CSV, and it is faster than reading the whole Parquet file as well. The reader consulted the metadata at the end of the file, found where the `revenue` column is stored, and read that part **only**. The other nine columns were never touched. This is called **column pruning**, and it is the practical payoff of the column layout of Section 1.

## 4. Why columnar files are so small

Keeping a column together has a second benefit. The values of one column resemble each other: they have the same type, and often only a few distinct values, frequently repeated in runs. Similar values next to each other compress extremely well. Two simple encodings do most of the work, and both can be written in a few lines.

### Dictionary encoding
The `city` column has a quarter of a million entries and only eight different values. Why spell out `Thessaloniki` thousands of times? Store the eight names **once**, in a small dictionary, and replace each entry by a small number.

In [ ]:
def dictionary_encode(values):
    dictionary = []
    position = {}
    codes = []
    for value in values:
        if value not in position:
            position[value] = len(dictionary)
            dictionary.append(value)
        codes.append(position[value])
    return dictionary, codes

dictionary, codes = dictionary_encode(columns["city"])
print("Dictionary:", dictionary)
print("First values:", columns["city"][:6])
print("First codes: ", codes[:6])

plain_size = sum(len(value) + 1 for value in columns["city"])            # the letters, plus a separator
encoded_size = len(codes) * 1 + sum(len(value) + 1 for value in dictionary)   # one byte per code, plus the dictionary
print(f"\nAs text:          {plain_size / 1000:8.0f} KB")
print(f"Dictionary-coded: {encoded_size / 1000:8.0f} KB   ({plain_size / encoded_size:.0f} times smaller)")

With eight values a code fits in a single byte, and in fact in three **bits**, so the real saving is larger still.

### Run-length encoding
Our rows are in order of date, so the `year` column is 2023 over a hundred thousand times, followed by 2024 over a hundred thousand times. Instead of repeating a value, write it once with the number of times it repeats.

In [ ]:
def run_length_encode(values):
    runs = []
    for value in values:
        if runs and runs[-1][0] == value:
            runs[-1][1] += 1
        else:
            runs.append([value, 1])
    return runs

for name in ["year", "month", "date", "city"]:
    runs = run_length_encode(columns[name])
    print(f"{name:<6} {len(columns[name]):>8,} values -> {len(runs):>8,} runs   first runs: {runs[:3]}")

The whole `year` column collapses into two runs, and `month` into twenty-four. For `city` the encoding achieves very little: neighbouring orders come from different cities, so the runs are only a few values long.

That last line holds an important lesson. **How well a column compresses depends on the order of the rows.** Let us measure it on real files: the same table written once in order of date and once in random order.

In [ ]:
shuffled_positions = list(range(len(rows)))
random.Random(1).shuffle(shuffled_positions)
shuffled_table = table.take(pa.array(shuffled_positions))

sorted_path = os.path.join(folder, "sorted.parquet")
shuffled_path = os.path.join(folder, "shuffled.parquet")
plain_path = os.path.join(folder, "uncompressed.parquet")

pq.write_table(table, sorted_path)
pq.write_table(shuffled_table, shuffled_path)
pq.write_table(table, plain_path, compression="none", use_dictionary=False)

names = ["CSV", "Parquet, no encoding", "Parquet, random order", "Parquet, sorted by date"]
sizes = [os.path.getsize(path) / 1_000_000 for path in [csv_path, plain_path, shuffled_path, sorted_path]]

plt.figure(figsize=(7, 3.2))
plt.barh(names[::-1], sizes[::-1])
plt.title("The same 250,000 rows on disk")
plt.xlabel("Megabytes")
plt.grid(True, axis="x")
plt.tight_layout()
plt.show()

for name, size in zip(names, sizes):
    print(f"{name:<26} {size:5.1f} MB")

The data is identical in all four files. Read the bars from the top.

The second bar may surprise you: a Parquet file written **without** any encoding is larger than the CSV. In binary, every whole number takes eight bytes, whether it is 3 or 3,000,000, while as text the digit `3` takes one. The column layout by itself saves nothing. What it does is put similar values next to each other, and **that** is what the encodings exploit: with them switched on, the file shrinks to a small fraction. A helpful order of rows then gains a little more.

## 5. Not reading at all: statistics

A Parquet file is divided into **row groups**, each holding a batch of rows. For every column of every row group the file records the smallest and the largest value. A reader can use these **statistics** to decide that a whole row group cannot possibly contain what a query wants, and skip it without reading a byte of it.

In [ ]:
grouped_path = os.path.join(folder, "grouped.parquet")
pq.write_table(table, grouped_path, row_group_size=25000)

metadata = pq.ParquetFile(grouped_path).metadata
date_column = column_names.index("date")
city_column = column_names.index("city")

print(metadata.num_row_groups, "row groups\n")
print(f"{'group':>5} {'rows':>7}   {'date: smallest .. largest':<28} {'city: smallest .. largest'}")
for group in range(metadata.num_row_groups):
    dates = metadata.row_group(group).column(date_column).statistics
    cities = metadata.row_group(group).column(city_column).statistics
    print(f"{group:>5} {metadata.row_group(group).num_rows:>7}   {dates.min} .. {dates.max}      {cities.min} .. {cities.max}")

Because the rows are in order of date, each row group covers its own stretch of the calendar. A query for March 2024 can see from the statistics alone that only one or two groups are worth opening.

For `city` the statistics are useless: every group contains everything from Athens to Volos. Statistics help only for columns whose values are **clustered** in the file. Once again the order of the rows decides.

In [ ]:
def groups_to_read(first_date, last_date):
    """The row groups whose range of dates overlaps the range we want."""
    needed = []
    for group in range(metadata.num_row_groups):
        statistics = metadata.row_group(group).column(date_column).statistics
        if statistics.max >= first_date and statistics.min <= last_date:
            needed.append(group)
    return needed

needed = groups_to_read("2024-03-01", "2024-03-31")
print(f"Row groups that can contain March 2024: {needed}")
print(f"Skipped without being read: {metadata.num_row_groups - len(needed)} of {metadata.num_row_groups}")

candidates = pq.ParquetFile(grouped_path).read_row_groups(needed, columns=["date", "revenue"])
by_hand = sum(revenue for date, revenue in zip(candidates["date"].to_pylist(), candidates["revenue"].to_pylist())
              if "2024-03-01" <= date <= "2024-03-31")
print(f"\nRows read: {candidates.num_rows:,} of {metadata.num_rows:,}")
print(f"Revenue in March 2024: {by_hand:,.2f}")

We read a fifth of the file and ignored the rest, on the strength of twenty small numbers in the metadata. The library does the same when given the condition as a `filters` argument.

In [ ]:
march = [("date", ">=", "2024-03-01"), ("date", "<=", "2024-03-31")]
filtered = pq.read_table(grouped_path, columns=["revenue"], filters=march)

print(f"{filtered.num_rows:,} rows, revenue {pc.sum(filtered['revenue']).as_py():,.2f}")

On a file of a few megabytes the time saved is too small to measure reliably. On files of many gigabytes, read over a network, skipping four fifths of the data is the difference between waiting seconds and waiting minutes.

Handing the condition to the reader, so that it can skip data at the source, is called **predicate pushdown**. Compare it with the index of Notebook 03. Both avoid reading rows that cannot match. An index is a separate structure, exact, and costly to maintain. Statistics are a few numbers per chunk, approximate, and almost free. Analytical systems, which read large ranges and not single rows, prefer the cheap approximate tool.

## 6. Partitioning: folders as an index

The same idea can be applied one level up. Instead of one large file, write **many files**, sorted into folders by the value of a column. A query that mentions that column then opens only the folders it needs.

In [ ]:
partitioned_folder = os.path.join(folder, "sales_by_month")
pq.write_to_dataset(table, root_path=partitioned_folder, partition_cols=["year", "month"])

paths = []
for directory, subfolders, files in os.walk(partitioned_folder):
    for name in files:
        paths.append(os.path.relpath(os.path.join(directory, name), partitioned_folder))

print(len(paths), "files. The first few:")
for path in sorted(paths)[:4]:
    print("  ", path.replace(os.sep, "/"))

The values of the partitioning columns have moved out of the files and into the **names of the folders**: `year=2024/month=3/`. The layout is a convention understood by nearly every analytical tool, including Spark.

In [ ]:
one_month = pq.read_table(partitioned_folder, columns=["revenue"], filters=[("year", "=", 2024), ("month", "=", 3)])

print(f"March 2024 from the partitioned folder: {one_month.num_rows:,} rows, read from 1 file out of {len(paths)}")
print(f"Total: {pc.sum(one_month['revenue']).as_py():,.2f}")

> ⚠️ **Common pitfalls**
>
> - **Partitioning by a column with too many values.** Partition by customer and you get five thousand folders of tiny files. Opening each file has a fixed cost, and thousands of tiny files are far slower than a few large ones. Partition by something with tens or hundreds of values, such as a month, and aim for files of many megabytes.
> - Using CSV for data that will be analysed repeatedly. Convert it once, and every later query benefits.
> - Expecting statistics or dictionary encoding to help on a column of unique, unordered values, such as an identifier. There is nothing to exploit.
> - Treating Parquet as a database. A Parquet file cannot be changed in place. To alter one row you write the file again. It is a format for data that is written once and read many times.

## 7. Where we stand

| | CSV | A row-store database | Parquet |
|---|---|---|---|
| Layout | rows, as text | rows, in binary | columns, encoded and compressed |
| Types recorded | no | yes | yes |
| Size on disk | largest | medium | smallest |
| Read a few columns of everything | slow | slow | **fast** |
| Fetch or change one record | slow | **fast** | not possible in place |
| A person can read it | **yes** | no | no |
| Best for | handing a small table to someone | running an application | analysing large data |

The question of the course was: what changes when the data no longer fits? Here the data stopped fitting the **shape of the work**. Analysis reads a few columns of everything, so we turned the table on its side, and then, column by column, squeezed out the repetition.

Through all of this the data stayed a table: every row with the same columns. The next notebook meets data that refuses that discipline.

---
## ✏️ Exercises

### Exercise 1 (Rows and columns by hand)
Using the Python structures `rows` and `columns`, compute the number of order lines in the category `Travel` in both layouts, and time each with `best_of`. Which is faster, and why?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
category_position = column_names.index("category")

from_rows, time_from_rows = best_of(lambda: sum(1 for row in rows if row[category_position] == "Travel"))
from_columns, time_from_columns = best_of(lambda: columns["category"].count("Travel"))

print(f"Rows:    {from_rows:,} in {time_from_rows:.2f} ms")
print(f"Columns: {from_columns:,} in {time_from_columns:.2f} ms")
```

*The question concerns one column only. In the column layout it is a single pass over one list. In the row layout every record must be opened to look at one of its ten values.*
</details>

### Exercise 2 (Which columns compress?)
Apply `dictionary_encode` to each of the columns `category`, `title`, `customer` and `order_id`, and print for each the number of distinct values and the ratio *number of values / number of distinct values*. For which columns is dictionary encoding worthwhile?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for name in ["category", "title", "customer", "order_id"]:
    dictionary, codes = dictionary_encode(columns[name])
    print(f"{name:<10} {len(dictionary):>8,} distinct values, each used {len(codes) / len(dictionary):>9,.1f} times on average")
```

*A column with 8 values used thirty thousand times each is ideal. A column in which almost every value is different, such as an identifier, gains nothing: the dictionary would be as large as the column. The number of distinct values in a column is called its cardinality, and it decides how a column should be stored.*
</details>

### Exercise 3 (Run lengths and order)
Sort the rows by `city` and then by `date`, rebuild the `city` and `year` columns from the sorted rows, and run-length encode both. Compare the number of runs with the figures of Section 4. What did the new order gain and what did it lose?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
by_city = sorted(rows, key=lambda row: (row[column_names.index("city")], row[column_names.index("date")]))

for name in ["city", "year"]:
    values = [row[column_names.index(name)] for row in by_city]
    print(f"{name:<5} sorted by city, then date: {len(run_length_encode(values)):>7,} runs"
          f"     sorted by date only: {len(run_length_encode(columns[name])):>7,} runs")
```

*The city column falls from tens of thousands of runs to eight. The year column goes from two runs to sixteen, two for each city. A table can be sorted in only one order, so the first sort column compresses best. The usual choice is the column that queries filter on most often, since the same order also makes the statistics of Section 5 useful.*
</details>

### Exercise 4 (Read less)
Using `pq.read_table` on `parquet_path`, compute the total `quantity` sold in the category `Science`, reading **only** the columns you need and passing the condition as a filter. Then check the answer with the Python list `rows`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
science = pq.read_table(parquet_path, columns=["quantity"], filters=[("category", "=", "Science")])
print("From Parquet:", pc.sum(science["quantity"]).as_py())

print("From the list:", sum(row[column_names.index("quantity")] for row in rows
                            if row[column_names.index("category")] == "Science"))
```

*The filter mentions `category`, but that column is not in the list of columns to return, so only `quantity` comes back. The reader fetched two columns out of ten.*
</details>

### Exercise 5 (Too many partitions, a little harder)
Write the table as a dataset partitioned by `city` into one folder, and by `date` into another. For each, print the number of files and their average size in kilobytes. Then time reading the whole of each dataset back. What do you conclude? (If you try to partition by `customer`, PyArrow will refuse: by default it will not create more than 1,024 partitions. Why might its authors have set such a limit?)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def describe(root):
    sizes = []
    for directory, subfolders, files in os.walk(root):
        for name in files:
            sizes.append(os.path.getsize(os.path.join(directory, name)))
    return len(sizes), sum(sizes) / len(sizes) / 1000

for column in ["city", "date"]:
    root = os.path.join(folder, f"by_{column}")
    pq.write_to_dataset(table, root_path=root, partition_cols=[column])
    count, average = describe(root)
    result, elapsed = best_of(lambda: pq.read_table(root), repeats=2)
    print(f"partitioned by {column:<5} {count:>5,} files, average {average:>8.1f} KB, read back in {elapsed:>8.1f} ms")
```

*Partitioning by date produces hundreds of files of a few kilobytes each, and reading them back is far slower than reading eight larger files, although the data is the same. Each file has its own metadata and must be found, opened and closed separately. This is known as the small-files problem, and it is one of the commonest ways to make a large data system slow. The limit of 1,024 partitions is there to stop you doing it by accident.*
</details>

## 🔑 Key takeaways

- A table can be laid out by **rows** or by **columns**. Rows suit fetching whole records, columns suit reading a few fields of every record.
- **CSV** is readable and universal, and it loses the types, must be read in full, and is bulky.
- **Parquet** stores columns with a schema. A reader takes only the columns it needs: **column pruning**.
- Columns compress well because their values are alike. **Dictionary encoding** replaces repeated values with small codes, and **run-length encoding** collapses runs.
- How well a file compresses, and how much can be skipped, depends on the **order of the rows**.
- **Statistics** per row group and **partitioning** into folders let a reader skip data without reading it: **predicate pushdown**.
- Too many small files are slow. Partition by columns with few values.

---
**Next:** *Notebook 06: NoSQL*, where the data stops being a tidy table: records with different fields, lists inside records, and databases that give up some of the guarantees of Module A in exchange for flexibility and scale.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*